# DS2002 · Pandas Challenge

**Lab — 2026-09-18 · Fall 2026**  

---

## Lab 04 — Pandas Challenge

Four hundred generated orders. Each question builds toward a demand report you could hand a vendor.

The data is seeded, so everyone's numbers should match. That is deliberate: if your total revenue differs from your neighbor's, one of you has a bug, and the assertions at the end will tell you which.

Every answer needs the number **and** a sentence saying what it means. A cell that prints `4218.5` with no interpretation is half an answer.

In [1]:
import pandas as pd, numpy as np
rng = np.random.default_rng(4)
n = 400
df = pd.DataFrame({
    'vendor_id': rng.choice(['V-01','V-05','V-10','V-18'], n),
    'category': rng.choice(['Food','Merch','RainGear','Drink'], n, p=[.5,.2,.1,.2]),
    'qty': rng.integers(1, 4, n),
    'price': rng.choice([4.5, 6.0, 7.5, 12.0, 24.0], n),
})
df.head()

,vendor_id,category,qty,price
0,V-10,Drink,2,24.0
1,V-18,RainGear,1,12.0
2,V-18,Drink,3,4.5
3,V-10,Food,2,12.0
4,V-18,Drink,3,7.5


### Q1 — Add `revenue`, then report total revenue and total units.

*Expected: 400 rows, and revenue should land between $8,000 and $9,000.*

In [2]:
df["revenue"] = df["qty"] * df["price"]
print("rows:", df.shape[0], "revenue:", df["revenue"].sum())

rows: 400 revenue: 8520.0


I calculated revenue by multiply quantity by price, then summing all revenue categories to get the total revenue.  The shape command gives us the number of rows and categories, so the first element should give us the number of rows.

### Q2 — Revenue by category, highest to lowest.

Include the share of total as a percentage in the same table.

In [3]:
total = df["revenue"].sum()
by_category = df.groupby("category")[["revenue"]].sum().sort_values("revenue",ascending=False)
by_category["share%"] = by_category["revenue"]/total*100
print(by_category)

          revenue     share%
category                    
Food       4293.0  50.387324
Merch      1771.5  20.792254
Drink      1554.0  18.239437
RainGear    901.5  10.580986


I grouped the data by the category they were classified under, then summed up the revenue for each category, in descending order.  To calculate the share of the total revenue, I added a new column that divided each category's revenue by the total revenue.

### Q3 — Which vendor has the highest *average* order revenue?

Report the average alongside the order count for each vendor. A high average on twelve orders is a different claim from a high average on two hundred.

In [4]:
df.groupby("vendor_id")["revenue"].agg(["mean","count"])

,mean,count
vendor_id,,
V-01,22.595745,94
V-05,20.580645,93
V-10,20.314286,105
V-18,21.750000,108


I grouped the original dataset by vendor_id, then summarized by the mean and row counts for each category.  Assuming that each row represents an order, the row count gives the total order count and the mean gives average revenue.

### Q4 — What share of revenue comes from Merch?

Print it as a percentage rounded to one decimal.

In [5]:
revenues = df.groupby("category")["revenue"].sum()
print(f"{revenues.loc["Merch"]/df["revenue"].sum():.1%}")

20.8%


Grouping by category type then summing up the revenues for each category gives the total revenue for each category.  Dividing the revenue for the Merch category by total revenue gives us how much Merch contributed.

### Q5 — Join in the vendor names.

The frame only has `vendor_id`. Merge the lookup below so your report is readable.

**Requirements:** left join, `validate='many_to_one'`, and prove the row count and revenue total did not change. One vendor id in the orders is not in this lookup — find it, and decide what to do about it.

In [6]:
vendor_names = pd.DataFrame({
    'vendor_id': ['V-01', 'V-05', 'V-10'],
    'vendor_name': ['Hoos Burgers', 'Rotunda Tacos', 'Cav Merch North'],
})

# TODO: merge, validate, and report the unmatched vendor
merged = df.merge(vendor_names,on="vendor_id",how="left",validate="many_to_one")
print("merged rows:", merged.shape[0], "merged revenue:", merged["revenue"].sum())
print("unmatched", merged.loc[merged["vendor_name"].isna(),"vendor_id"].unique())
merged.loc[merged["vendor_id"]=="V-18", 'vendor_name'] = "unknown"
print(merged.head())

merged rows: 400 merged revenue: 8520.0
unmatched ['V-18']
  vendor_id  category  qty  price  revenue      vendor_name
0      V-10     Drink    2   24.0     48.0  Cav Merch North
1      V-18  RainGear    1   12.0     12.0          unknown
2      V-18     Drink    3    4.5     13.5          unknown
3      V-10      Food    2   12.0     24.0  Cav Merch North
4      V-18     Drink    3    7.5     22.5          unknown


**The unmatched vendor, and what I did about it:** V-18, and I assigned it to be unknown under the vendor_name.


I merged df and vendor_names through the vendor_id category, then printed the rows and revenue total from the result.

### Q6 — A pivot table: vendors down the side, categories across the top, revenue in the cells.

Add row and column totals so it reads as a report rather than a grid of numbers.

In [7]:
pd.pivot_table(merged,values="revenue",index="vendor_name",columns="category",aggfunc="sum",margins=True)

category,Drink,Food,Merch,RainGear,All
vendor_name,,,,,
Cav Merch North,502.5,1054.5,400.5,175.5,2133.0
Hoos Burgers,171.0,1338.0,373.5,241.5,2124.0
Rotunda Tacos,298.5,882.0,489.0,244.5,1914.0
unknown,582.0,1018.5,508.5,240.0,2349.0
All,1554.0,4293.0,1771.5,901.5,8520.0


I used the built-in pivot table function, where the values in each cell was the revenue, indexed by the vendor names.

### Q7 — Validate your work

**TODO:** uncomment and make these pass. Assign your results to the named variables as you go.

In [8]:
assert len(df) == 400
assert 8000 < df['revenue'].sum() < 9000, df['revenue'].sum()
assert abs(by_category['revenue'].sum() - df['revenue'].sum()) < 0.01
assert len(merged) == len(df), 'the vendor merge changed the row count'
print('checks passed.')

checks passed.


### Write-up

**a)** What would you tell these vendors to do differently next game? One paragraph, with at least two numbers from your report in it.

**b)** Which of your seven answers is the least trustworthy, and why? Point at a specific weakness — a small group size, an unmatched vendor, a category that is really two things.

a: I would tell these vendors to strategize on how to sell more drinks or on how to price their drinks higher.  Aside from rain gear, which is conditional on weather conditions, drinks made the least revenue of different categories.  While food made about 50.4% of the total revenue, drinks only made up 18.3%.  This suggests that people are willing and able to purchase concessions, but food is bringing in much more money for some reason.

b: I think my answer to #5, the join question is the least trustworthy.  This is because there is an unmatched vendor: there is no name correlated to vendor 18, which could indicate that a mistake has been made somewhere with the logging of the data.